In [50]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import mutual_info_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_validate
from sklearn.metrics import balanced_accuracy_score
from imblearn.pipeline import make_pipeline
from imblearn.under_sampling import RandomUnderSampler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

In [3]:
home = Path.home()
data_path = home / 'Programming/data/fraud-detection/data'
data_df = pd.read_csv(data_path / 'transactions_obf.csv',
                      parse_dates=['transactionTime'],
                      dtype={'availableCash': np.int64,
                             'transactionAmount': np.float64})
data_df.sort_values(by='transactionTime', inplace=True)

In [4]:
labels_df = pd.read_csv(data_path / 'labels_obf.csv', parse_dates=['reportedTime'])
labels_df.sort_values(by='reportedTime', inplace=True)

In [5]:
print(data_df.info())

<class 'pandas.core.frame.DataFrame'>
Index: 118621 entries, 0 to 118619
Data columns (total 10 columns):
 #   Column             Non-Null Count   Dtype              
---  ------             --------------   -----              
 0   transactionTime    118621 non-null  datetime64[ns, UTC]
 1   eventId            118621 non-null  object             
 2   accountNumber      118621 non-null  object             
 3   merchantId         118621 non-null  object             
 4   mcc                118621 non-null  int64              
 5   merchantCountry    118621 non-null  int64              
 6   merchantZip        95616 non-null   object             
 7   posEntryMode       118621 non-null  int64              
 8   transactionAmount  118621 non-null  float64            
 9   availableCash      118621 non-null  int64              
dtypes: datetime64[ns, UTC](1), float64(1), int64(4), object(4)
memory usage: 10.0+ MB
None


In [6]:
data_df[['eventId', 'accountNumber', 'merchantId', 'mcc', 'merchantCountry', 'merchantZip', 'posEntryMode']] = data_df[['eventId', 'accountNumber', 'merchantId', 'mcc', 'merchantCountry', 'merchantZip', 'posEntryMode']].astype('string')

In [7]:
print(data_df.describe())

       transactionAmount  availableCash
count      118621.000000  118621.000000
mean           53.674774    6625.508974
std           183.665315    3410.289486
min            -0.150000     500.000000
25%             8.030000    4500.000000
50%            20.250000    7500.000000
75%            49.000000    8500.000000
max         13348.000000   18500.000000


In [8]:
data_df.head()

,transactionTime,eventId,accountNumber,merchantId,mcc,merchantCountry,merchantZip,posEntryMode,transactionAmount,availableCash
0,2017-01-01 00:00:00+00:00,18688431A1,94f9b4e7,b76d06,5968,826,CR0,1,10.72,7500
3,2017-01-01 00:15:07+00:00,11162049A1,038099dd,7d5803,5499,826,NR1,81,21.00,7500
4,2017-01-01 00:37:09+00:00,17067235A1,3130363b,12ca76,5411,826,M50,81,47.00,10500
2,2017-01-01 00:43:17+00:00,31294145A1,c0ffab1b,94cafc,5735,442,<NA>,81,5.04,9500
1,2017-01-01 00:49:03+00:00,2164986A1,648e19cf,718cc6,5499,826,DE14,81,21.19,4500


In [9]:
print(f'Number of unique categorical entries:\n'
      f'{data_df.select_dtypes('string').nunique().sort_values(ascending=False)}')

Number of unique categorical entries:
eventId            118621
merchantId          33327
merchantZip          3260
accountNumber         766
mcc                   361
merchantCountry        82
posEntryMode           10
dtype: int64


In [10]:
print(f'Number of missing values:\n{data_df.isnull().sum()}')

Number of missing values:
transactionTime          0
eventId                  0
accountNumber            0
merchantId               0
mcc                      0
merchantCountry          0
merchantZip          23005
posEntryMode             0
transactionAmount        0
availableCash            0
dtype: int64


In [11]:
print(f'Percentage of missing values:\n{round(100*data_df.isnull().sum()/data_df.shape[0], 1)}')

Percentage of missing values:
transactionTime       0.0
eventId               0.0
accountNumber         0.0
merchantId            0.0
mcc                   0.0
merchantCountry       0.0
merchantZip          19.4
posEntryMode          0.0
transactionAmount     0.0
availableCash         0.0
dtype: float64


In [12]:
print(f'Sorted list of normalized merchantZip codes:\n{data_df.merchantZip.value_counts(dropna=False, normalize=True)[:10]}')

Sorted list of normalized merchantZip codes:
merchantZip
<NA>    0.193937
0       0.122019
E12     0.009644
SL4     0.005496
LS11    0.005151
CO10    0.004637
HA8     0.004578
AL10    0.004578
NW4      0.00446
IV30    0.004426
Name: proportion, dtype: Float64


In [13]:
print(f'''Percentage of NAs and 0s in 'merchantZip': {round(100*((data_df.merchantZip == '0').sum() + data_df.merchantZip.isna().sum())/data_df.shape[0], 2)}%.''')

Percentage of NAs and 0s in 'merchantZip': 31.6%.


In [14]:
data_df['merchantZip'] = data_df['merchantZip'].replace({np.nan: 'Unknown', '0': 'Unknown'})

In [15]:
print(f'Sorted list of normalized merchantZip codes:\n{data_df.merchantZip.value_counts(dropna=False, normalize=True)[:10]}')

Sorted list of normalized merchantZip codes:
merchantZip
Unknown    0.315956
E12        0.009644
SL4        0.005496
LS11       0.005151
CO10       0.004637
HA8        0.004578
AL10       0.004578
NW4         0.00446
IV30       0.004426
TD1        0.003861
Name: proportion, dtype: Float64


In [16]:
print(f'Sorted list of normalized posEntryMode codes:\n{data_df.posEntryMode.value_counts(dropna=False, normalize=True)[:10]}')

Sorted list of normalized posEntryMode codes:
posEntryMode
5     0.592037
81    0.302071
1      0.08909
90    0.010133
7      0.00537
80    0.000776
79    0.000261
2     0.000135
0     0.000093
91    0.000034
Name: proportion, dtype: Float64


In [17]:
print(labels_df.info())

<class 'pandas.core.frame.DataFrame'>
Index: 875 entries, 0 to 874
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype              
---  ------        --------------  -----              
 0   reportedTime  875 non-null    datetime64[ns, UTC]
 1   eventId       875 non-null    object             
dtypes: datetime64[ns, UTC](1), object(1)
memory usage: 20.5+ KB
None


In [18]:
labels_df['eventId'] = labels_df['eventId'].astype('string')

In [19]:
print(f'Number of unique entries in labels dataset:\n{labels_df.nunique()}')

Number of unique entries in labels dataset:
reportedTime    145
eventId         875
dtype: int64


In [20]:
data_df['fraudCase'] = data_df.eventId.isin(labels_df.eventId).astype(bool)

In [21]:
neg, pos = np.bincount(data_df['fraudCase'])
total = neg + pos
print(
    f'Total: {total}\n'
    f'Negative fraud cases: {neg} ({100 * neg / total:.2f}% of total)\n'
    f'Positive fraud cases: {pos} ({100 * pos / total:.2f}% of total)\n'
    )


Total: 118621
Negative fraud cases: 117746 (99.26% of total)
Positive fraud cases: 875 (0.74% of total)



In [22]:
# information on the data set
total_num_accounts = data_df['accountNumber'].nunique()
fraud_df = data_df.loc[data_df.fraudCase == 1]
grouped = fraud_df.groupby('accountNumber')['transactionAmount']
fraud_amount_per_account_df = grouped.sum()
num_accounts_with_fraud = len(fraud_amount_per_account_df)
num_accounts_frauds_less_than_1000 = np.sum(fraud_amount_per_account_df < 1000)

In [23]:
results_dict ={
    'total_num_accounts': total_num_accounts,
    'num_accounts_with_fraud': num_accounts_with_fraud,
    'percent_fraud_per_transaction':np.round(
        ((data_df['fraudCase'] == 1).sum()/data_df.shape[0])*100, 2
        ),
    'percent_accounts_with_fraud': np.round(
        100*num_accounts_with_fraud/total_num_accounts, 2
        ),
    'percent_frauds_less_than_1000_GBP': np.round(
        (num_accounts_frauds_less_than_1000/num_accounts_with_fraud)*100, 2),
}

In [24]:
print(f'''Total number of accounts: {
    results_dict['total_num_accounts']}'''
    )
print(f'''Number of accounts with fraud: {
    results_dict['num_accounts_with_fraud']}'''
    )
print(f'''Percent of fraud per transaction: {
    results_dict['percent_fraud_per_transaction']}%'''
    )
print(f'''Percent of accounts with fraud: {
    results_dict['percent_accounts_with_fraud']}%'''
    )
print(f'''Percent of fraud in accounts with less than £1000: {
    results_dict['percent_frauds_less_than_1000_GBP']}%'''
    )

Total number of accounts: 766
Number of accounts with fraud: 167
Percent of fraud per transaction: 0.74%
Percent of accounts with fraud: 21.8%
Percent of fraud in accounts with less than £1000: 83.23%


In [25]:
def mutual_info(x):
    return mutual_info_score(x, data_df.fraudCase)

In [26]:
categorical_list = list(data_df.select_dtypes('string'))
numerical_list = list(data_df.select_dtypes('number'))

In [27]:
print(f'Mutual information score:\n{data_df[categorical_list].apply(mutual_info).sort_values(ascending=False)}')

Mutual information score:
eventId            0.043564
merchantId         0.031126
accountNumber      0.020102
merchantZip        0.008898
mcc                0.008623
merchantCountry    0.006532
posEntryMode       0.006042
dtype: float64


In [28]:
corr_df = data_df[numerical_list].corr()
print(f'Correlation matrix of numerical features:\n{corr_df}')

Correlation matrix of numerical features:
                   transactionAmount  availableCash
transactionAmount           1.000000       0.063817
availableCash               0.063817       1.000000


In [29]:
data_df.drop(['eventId', 'transactionTime', 'merchantId'], axis=1, inplace=True)
updated_categorical_list = list(data_df.select_dtypes('string'))

In [30]:
selected_list = updated_categorical_list + numerical_list
dicts = data_df[selected_list].to_dict(orient='records')
dv = DictVectorizer(sparse=False)
dicts_arr = dv.fit_transform(dicts)
print(f'Shape of processed dataframe: {dicts_arr.shape}')

Shape of processed dataframe: (118621, 4481)


In [31]:
feature_names_list = dv.get_feature_names_out(selected_list)

In [32]:
processed_df = pd.DataFrame(dicts_arr, columns=feature_names_list)

In [33]:
y = data_df['fraudCase']
X = processed_df
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
    )

In [51]:
model_dict = {'logistic regression': LogisticRegression(random_state=0, max_iter=4000),
              'decision tree': DecisionTreeClassifier(max_depth=6, random_state=0),
              'random forest': RandomForestClassifier(n_jobs=-1, random_state=0,
                                                      max_depth=6, n_estimators=150),
              'histogram gradient boosting': HistGradientBoostingClassifier(
                  max_depth=8, max_iter=400, random_state=0),
            }

In [52]:
for name, model in model_dict.items():
    pipeline = make_pipeline(RandomUnderSampler(random_state=0), model)
    cv_res = cross_validate(
        pipeline, X_train, y_train, scoring="balanced_accuracy",
        return_train_score=True, return_estimator=True,
        n_jobs=-1, error_score='raise'
        )
    print(f"Balanced accuracy mean ± std. dev. for {name}: "
          f"{np.round(cv_res['test_score'].mean(), 3)} ± "
          f"{np.round(cv_res['test_score'].std(), 3)}")
    scores = []
    for cv_model in cv_res["estimator"]:
        scores.append(balanced_accuracy_score(y_test, cv_model.predict(X_test)) )
    print(f"Balanced accuracy mean ± std. dev. for {name} on test dataset: "
          f"{np.round(np.mean(scores), 3)} ± "
          f"{np.round(np.std(scores), 3)}"
          f"\n")

Balanced accuracy mean ± std. dev. for logistic regression: 0.899 ± 0.011
Balanced accuracy mean ± std. dev. for logistic regression on test dataset: 0.899 ± 0.005

Balanced accuracy mean ± std. dev. for decision tree: 0.839 ± 0.013
Balanced accuracy mean ± std. dev. for decision tree on test dataset: 0.846 ± 0.004

Balanced accuracy mean ± std. dev. for random forest: 0.825 ± 0.011
Balanced accuracy mean ± std. dev. for random forest on test dataset: 0.836 ± 0.008

Balanced accuracy mean ± std. dev. for histogram gradient boosting: 0.868 ± 0.018
Balanced accuracy mean ± std. dev. for histogram gradient boosting on test dataset: 0.86 ± 0.004

